# Day 2 - Code breaks (run during the lecture)

Short pieces of code that go with the slides. Each break takes 3-5 minutes:
**run** the cell, look at the output, then do the **🔧 Change it** tasks.

- Open this notebook in Colab (link in the course repository). A **CPU runtime is enough** - no GPU needed.
- Run the **setup cell** first (the next code cell). After a restart, run it again.
- You cannot break anything: if something looks strange, *Runtime → Restart session* and start again.

The setup cell loads PyTorch (`torch`), the building blocks for networks (`nn`), ready-made functions such as
loss functions (`F`) and the plotting library (`plt`).

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)  # same random numbers for everybody
print("ready - PyTorch", torch.__version__)

## 💻 Code break 1 - Loss and gradient in a few lines
The house prices from "train your dragon": one neuron, the MSE loss, its slopes (chain rule) and 20 steps of
gradient descent.

In [ ]:
x = torch.tensor([2.104, 1.600, 2.400])   # area in 1000 sq ft
y = torch.tensor([399.9, 329.9, 369.0])   # price in 1000 $

# plot the data
plt.scatter(x, y)
plt.xlabel("area in 1000 sq ft")
plt.ylabel("price in 1000 $")

In [ ]:
# now lets try to fit a line through the data points
w, b, lr = 0.0, 0.0, 0.1
for step in range(20):
    e = w * x + b - y                     # error per house
    loss = (e ** 2).mean()                # MSE
    dw = (2 * e * x).mean()               # chain rule: 2 * error * input
    db = (2 * e).mean()
    w = w - lr * dw                       # gradient descent
    b = b - lr * db                       # gradient descent
    print(step, round(loss.item(), 1))

🔧 **Change it:**
1. Change `lr = 0.1` to `lr = 0.2` and run again - what happens to the loss?
2. Change it to `lr = 0.01` - faster or slower?
3. Set `lr = 0.1` again and change `range(20)` to `range(200)`. How small does the loss get?

*What you should see:* with 0.1 the loss falls from about 135 000 to about 1 100 in 20 steps (about 600 after
200 steps; the best possible line has about 450). With 0.2 it explodes (about 3.7 million after 20 steps - the
steps are too large). With 0.01 it falls, but more slowly (about 3 300 after 20 steps).

In [ ]:
# plot the starting point and the fitted line
plt.scatter(x, y)
plt.plot(x, w * x + b, color="orange", label="fitted line")
plt.xlabel("area in 1000 sq ft")
plt.ylabel("price in 1000 $")

## 💻 Code break 2 - Activation functions in PyTorch
The three activation functions from the slides, applied to five numbers.

In [ ]:
z = torch.tensor([-3.0, -1.6, 0.0, 1.6, 3.0])
print(torch.relu(z))
print(torch.sigmoid(z))
print(torch.tanh(z))

The same three functions as curves from -4 to 4.

In [ ]:
z_curve = torch.linspace(-4, 4, 200)
plt.figure(figsize=(5, 3))
plt.plot(z_curve, torch.relu(z_curve), label="ReLU")
plt.plot(z_curve, torch.sigmoid(z_curve), label="sigmoid")
plt.plot(z_curve, torch.tanh(z_curve), label="tanh")
plt.axhline(0, color="gray", lw=0.5)
plt.legend()
plt.show()

🔧 **Change it:**
1. Add the number `10.0` to `z`. What do sigmoid and tanh give? (They "saturate".)
2. Replace `torch.relu(z)` by `F.gelu(z)`. What is different for negative numbers?

*What you should see:* ReLU sets every negative number to 0 and keeps positive ones; sigmoid squeezes everything
into 0...1, tanh into -1...1; for large numbers both are almost exactly 1.

## 💻 Code break 3 - Without an activation, layers stay a straight line
Two small networks with random weights: one without an activation function, one with ReLU in the middle.

In [ ]:
# the three layers of `linear`: (name, number of neurons)
layers = [
    ("Input", 1),
    ("Linear layer 1", 8),
    ("Output", 1),
]
# Nicer visualization of the network
fig, ax = plt.subplots(figsize=(11, 6))

layer_x = [0, 3, 6]
node_positions = {}

# Calculate neuron positions
for layer_index, (label, neuron_count) in enumerate(layers):
    ys = torch.linspace(
        -(neuron_count - 1) / 2,
        (neuron_count - 1) / 2,
        neuron_count
    )
    node_positions[layer_index] = [
        (layer_x[layer_index], float(y)) for y in ys
    ]

# Draw connections
for layer_index in range(len(layers) - 1):
    for x1, y1 in node_positions[layer_index]:
        for x2, y2 in node_positions[layer_index + 1]:
            ax.plot(
                [x1, x2], [y1, y2],
                color="lightgray",
                lw=0.7,
                alpha=0.7,
                zorder=1
            )

# Draw neurons and labels
for layer_index, (label, neuron_count) in enumerate(layers):
    positions = node_positions[layer_index]
    xs, ys = zip(*positions)

    ax.scatter(
        xs, ys,
        s=900,
        color="#76b5c5",
        edgecolor="#1f4e5f",
        linewidth=1.5,
        zorder=2
    )

    # Add neuron numbers
    for neuron_number, (xpos, ypos) in enumerate(positions, start=1):
        ax.text(
            xpos, ypos, str(neuron_number),
            ha="center",
            va="center",
            fontsize=9,
            color="white",
            weight="bold",
            zorder=3
        )

    # Add layer name and neuron count
    ax.text(
        layer_x[layer_index],
        4.5,
        label,
        ha="center",
        va="bottom",
        fontsize=12,
        weight="bold"
    )

    ax.text(
        layer_x[layer_index],
        -4.5,
        f"{neuron_count} neuron" if neuron_count == 1
        else f"{neuron_count} neurons",
        ha="center",
        va="top",
        fontsize=10,
        color="#444444"
    )

ax.set_xlim(-1, 7)
ax.set_ylim(-5.2, 5.2)
ax.set_title("Structure of the Neural Network", fontsize=15, pad=20)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Input data
x = torch.linspace(-3, 3, 100).reshape(-1, 1)        # 100 inputs, 1 feature

print("x shape:", x.shape)

In [ ]:
linear = nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 1))
bent   = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))
plt.plot(x, linear(x).detach(), label="no activation") # detach() is used to avoid tracking gradients for this operation - we need only the output values, not the gradients
plt.plot(x, bent(x).detach(), label="with ReLU")
plt.legend(); plt.show()

🔧 **Change it:**
1. Run the cell several times (new random weights each time). Is "no activation" ever not a straight line?
2. Add more layers to `linear`: `nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 8), nn.Linear(8, 1))`. Still straight?
3. Replace `nn.ReLU()` by `nn.Tanh()`.

*What you should see:* without an activation the output is **always** a straight line, no matter how many layers -
the layers "collapse". With ReLU the line has kinks; with tanh it is smoothly bent.

### 🔍 Look inside the ReLU network
`bent` has 8 hidden neurons. Neuron *i* computes `relu(w_i * x + b_i)` - a "hinge": 0 on one side, a straight line
on the other. The output layer multiplies every hinge by its output weight `v_i` and adds everything up:

`output = v_1 * relu(w_1 * x + b_1) + ... + v_8 * relu(w_8 * x + b_8) + c`

- **Left:** the 25 numbers of the network (8 + 8 in layer 1, 8 + 1 in layer 2), one color per neuron.
- **Right:** one thin line per neuron = its piece `v_i * relu(w_i * x + b_i)`. The dotted line marks its **kink**
  at `x = -b_i / w_i` (where `w_i * x + b_i = 0`). The black line is the sum of all pieces - the network output.

In [ ]:
# Look inside the ReLU network: each hidden neuron is a "hinge" relu(w*x + b),
# the output layer multiplies every hinge by its output weight and adds them up.
w_in, b_in = bent[0].weight.detach().flatten(), bent[0].bias.detach()     # layer 1: one weight + one bias per neuron
w_out, b_out = bent[2].weight.detach().flatten(), bent[2].bias.detach()   # layer 2: one weight per neuron + one bias
k = len(w_in)                                                             # number of hidden neurons
colors = [f"C{i}" for i in range(k)]                                      # one color per neuron, same in both plots

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# left: the parameters (random, the network is not trained)
for g, values in enumerate([w_in, b_in, w_out]):
    ax1.bar(g * (k + 2) + torch.arange(k), values, color=colors)
ax1.set_xticks([g * (k + 2) + i for g in range(3) for i in range(k)], [i + 1 for g in range(3) for i in range(k)])
for g, name in enumerate(["weight in (layer 1)", "bias (layer 1)", "weight out (layer 2)"]):
    ax1.text(g * (k + 2) + (k - 1) / 2, -0.12, name, ha="center", va="top", transform=ax1.get_xaxis_transform())
ax1.axhline(0, color="gray", lw=0.5)
ax1.set_title(f"{k} hidden neurons (output bias = {b_out.item():.2f})")

# right: what every neuron adds to the output
pieces = torch.relu(x * w_in + b_in) * w_out          # 100 x k: one column per neuron
for i in range(k):
    ax2.plot(x, pieces[:, i], color=colors[i], lw=1)
    kink = (-b_in[i] / w_in[i]).item()                 # the ReLU switches on/off where w*x + b = 0
    if x.min() < kink < x.max():
        ax2.axvline(kink, color=colors[i], ls=":", lw=1)
        ax2.text(kink, 1.01, str(i + 1), color=colors[i], ha="center", transform=ax2.get_xaxis_transform())
ax2.plot(x, pieces.sum(1) + b_out, color="black", lw=3, label="sum of all pieces + output bias")
ax2.plot(x, bent(x).detach(), color="orange", ls="--", lw=1.5, label="bent(x) - the network itself")
ax2.axhline(0, color="gray", lw=0.5)
ax2.set_xlabel("x")
ax2.legend()
plt.tight_layout(); plt.show()

🔧 **Change it:**
1. Run the network cell above again (new random weights), then this cell. Where are the kinks now?
2. Find the kinks in the black line. Every kink belongs to one neuron - which one? Can there be more than 8?
3. In the network cell, use 3 hidden neurons (`nn.Linear(1, 3)`, `nn.Linear(3, 1)`), then 32. Run both cells.

*What you should see:* the black line is exactly the orange dashed line - the network really is just a sum of
hinges. It is straight between the kinks and bends only where one neuron switches on or off, so there are at
most 8 kinks. More neurons = more kinks = more flexible curves.

### 🔍 One input, step by step: x = 2
What happens inside `bent` when we put in **one** number? Three steps, the same as the three parts of `bent`:
1. **Layer 1** (`bent[0]`): every hidden neuron computes `z = w * x + b`.
2. **ReLU** (`bent[1]`): negative `z` become 0 - those neurons are "off" for this input.
3. **Layer 2** (`bent[2]`): multiply every `relu(z)` by its output weight `v`, add everything up, add the bias `c`.

In [ ]:
# How does the input x get through the network? (Example with x = 2 and the ReLU network above)
# uses w_in, b_in, w_out, b_out, k and colors from the cell above
x0 = torch.tensor([[2.0]])         # one input: x = 2
z = bent[0](x0)                    # step 1 - layer 1: z = w * x + b for every hidden neuron
h = bent[1](z)                     # step 2 - ReLU: negative numbers become 0
out = bent[2](h)                   # step 3 - layer 2: v_1 * h_1 + ... + v_8 * h_8 + c
z, h, out = z.detach()[0], h.detach()[0], out.detach()[0, 0]

print("neuron |    w   * x +    b   =    z    | relu(z) |   * v    =  piece")
for i in range(k):
    piece = h[i] * w_out[i] + 0.0  # + 0.0 turns -0.0 into 0.0
    print(f"{i + 1:6} | {w_in[i]:6.3f} * {x0.item():g} + {b_in[i]:6.3f} = {z[i]:6.3f} | {h[i]:7.3f} | * {w_out[i]:6.3f} = {piece:6.3f}")
print(f"\nsum of pieces + c = {(h * w_out).sum():.3f} + {b_out.item():.3f} = {out:.3f}")
print(f"bent(x0)          = {bent(x0).item():.3f}")

# the same steps as pictures
n = torch.arange(1, k + 1)
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, values, title in zip(axes, [z, h, h * w_out], ["1) z = w * x + b", "2) h = relu(z)", "3) piece = v * h"]):
    ax.bar(n, values, color=colors)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_xticks(n)
    ax.set_xlabel("hidden neuron")
    ax.set_title(title)
axes[3].plot(x, bent(x).detach(), color="black")
axes[3].plot(x0.item(), out, "o", color="orange", ms=10)
axes[3].set_title(f"4) sum + c = {out:.2f} at x = {x0.item():g}")
axes[3].set_xlabel("x")
plt.tight_layout(); plt.show()

🔧 **Change it:**
1. Check one row of the table with a calculator (or by hand).
2. Change `x0` to `-2.0`. Which neurons are "off" now? Are they the same as for `x = 2`?
3. Change `x0` to `0.0`. What is `z` now? (Hint: `w * 0 = 0`.)

*What you should see:* `bent(x0)` gives exactly the number from the table - a network is nothing more than
"multiply, add, ReLU, multiply, add". Neurons with negative `z` are "off" and add nothing. For another `x`,
other neurons are off. With `x = 0` every `z` is just the bias `b`.

## 💻 Code break 4 - An MLP in PyTorch
The EEG network from the lab: 90 features in, two hidden layers with 64 neurons, 2 classes out.

In [ ]:
model = nn.Sequential(
    nn.Linear(90, 64), nn.ReLU(),     # hidden layer 1
    nn.Linear(64, 64), nn.ReLU(),     # hidden layer 2
    nn.Linear(64, 2))                 # output: 2 classes
print(model)
print(sum(p.numel() for p in model.parameters()))   # number of parameters
x = torch.randn(8, 90)                # a batch: 8 windows, 90 features
print(model(x).shape)                 # 8 rows, 2 outputs

🔧 **Change it:**
1. Make the hidden layers wider: `64` → `128` (in all three places). How many parameters now? (28 418)
2. Change the batch to `torch.randn(32, 90)`. What is the output shape?
3. Change it to `torch.randn(8, 80)`. Read the error message: which numbers do not fit?

*What you should see:* 10 114 parameters and `torch.Size([8, 2])` - one row per window, one score per class.
With 80 features you get `mat1 and mat2 shapes cannot be multiplied (8x80 and 90x64)`: the first layer expects 90.

## 💻 Code break 5 - Softmax and cross-entropy
The softmax example from the slides (cat, dog, horse) and the loss for two different true classes.

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.0]])          # one sample, 3 classes
print(torch.softmax(logits, dim=1))               # probabilities
print(F.cross_entropy(logits, torch.tensor([0]))) # true class 0 (cat)
print(F.cross_entropy(logits, torch.tensor([2]))) # true class 2 (horse)

🔧 **Change it:**
1. Make the network more sure about "cat": `logits = torch.tensor([[5.0, 1.0, 0.0]])`. What happens to both losses?
2. Make all logits equal: `[[1.0, 1.0, 1.0]]`. What are the probabilities?

*What you should see:* probabilities 0.67 / 0.24 / 0.09; loss 0.41 if the truth is cat, 2.41 if it is horse.
With `[[5.0, 1.0, 0.0]]`: 0.02 (cat) and 5.02 (horse). With equal logits every class gets 1/3.
Confident and right = small loss, confident and wrong = large loss.

## 💻 Code break 6 - Autograd on the worked example
The one-neuron example from the backprop slides. PyTorch computes the slopes for us.

In [ ]:
x, t = torch.tensor(3.0), torch.tensor(4.0)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
L = (torch.relu(w * x + b) - t) ** 2
L.backward()
print(L.item(), w.grad.item(), b.grad.item())

🔧 **Change it:**
1. Change `b` to `-7.0`. Why are both slopes 0 now? (Hint: what does ReLU do with a negative number?)
2. Set `b` back to `-1.0` and change `x` to `6.0`. What happens to the loss and the slopes?

*What you should see:* `1.0 6.0 2.0` - the same numbers as on the slide. With `b = -7` the ReLU blocks
everything: `16.0 0.0 0.0` (a "dead" ReLU). With `x = 6`: `49.0 84.0 14.0`.

## 💻 Code break 7 - Why zero_grad?
We compute the slope of the same loss three times - without deleting the old slope in between.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    L = (w * 3.0 - 4.0) ** 2
    L.backward()
    print(w.grad.item())          # the slopes add up!

🔧 **Change it:**
1. Put `w.grad = None` as the first line inside the loop (indented like the other lines). Run again.

*What you should see:* `12.0 24.0 36.0` - PyTorch **adds** each new slope to the old one. With `w.grad = None`:
`12.0 12.0 12.0`. That is why every training step starts with `optimizer.zero_grad()`.